# MLLM Teacher: Qwen2.5-Omni-3B on MIntRec 2.0

**Goal**: Download MIntRec 2.0 (text + audio + video, raw `.mp4` clips), explore the task, then run the frozen Qwen2.5-Omni-3B teacher over **video + audio + text** and verify hidden-state extraction.

This mirrors `src/early_experiment/mllm_teacher.ipynb` (FSC, audio-only) but uses the full tri-modal teacher input. Designed to run end-to-end on **RunPod** (Linux GPU, e.g. L4 24 GB); the 4-bit cell also fits a local 6 GB 3060 for a quick smoke test.

- Dataset source: **HuggingFace** [`THUIAR/MMLA-Datasets`](https://huggingface.co/datasets/THUIAR/MMLA-Datasets) → `MIntRec2.0/` (annotations + packed raw videos, ~9 GB). No Google-Drive quota headaches.
- Original benchmark: [thuiar/MIntRec2.0](https://github.com/thuiar/MIntRec2.0) (ICLR 2024) — 30 fine-grained intents from *Superstore / Big Bang Theory / Friends*.
- Annotation `id` = `{dialogue_id}_{utterance_id}` (e.g. `0_3`); raw clips are `.mp4`.

## 1. Setup & Installs

`huggingface_hub` pulls the dataset; `qwen-omni-utils` packs video+audio for the Omni processor. **On RunPod, do NOT run the repo `requirements.txt`** — it pins a Windows CUDA build of torch that would clobber the image's GPU-matched torch. We only add libraries on top of the base image's torch.

In [ ]:
%pip install -q huggingface_hub "qwen-omni-utils[decord]" librosa soundfile av
# Qwen2.5-Omni needs transformers>=4.52; bitsandbytes for 4-bit; accelerate for device_map.
%pip install -q "transformers>=4.52.0" accelerate bitsandbytes

In [ ]:
import os
import sys

# Register FFmpeg shared DLLs on Windows so torchcodec/av can find avcodec/avformat.
# The return value MUST be stored — if GC'd, the dir is removed from the DLL search path.
# On Linux (RunPod) this whole block is skipped; ffmpeg comes from the system.
if sys.platform == 'win32':
    _ffmpeg_dll_dir = None
    for _p in os.environ.get('PATH', '').split(';'):
        if _p and os.path.exists(os.path.join(_p, 'avcodec-62.dll')):
            _ffmpeg_dll_dir = os.add_dll_directory(_p)
            break
    if _ffmpeg_dll_dir is None:
        print('WARNING: avcodec-62.dll not found on PATH — video/audio decoding may fail')
    else:
        print(f'FFmpeg DLL dir registered: {_p}')

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Audio as IPyAudio, display

# Put repo src/ on path so `import common.config` resolves regardless of launch dir.
# This notebook lives at src/mintrec/early_experiment/ -> walk up to the 'src' dir.
_SRC = Path.cwd()
while _SRC.name != 'src' and _SRC != _SRC.parent:
    _SRC = _SRC.parent
if _SRC.name == 'src' and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from common.config import MINTREC_DATA, MINTREC_OUTPUTS

MINTREC_DATA.mkdir(parents=True, exist_ok=True)
MINTREC_OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f'PyTorch     : {torch.__version__}')
print(f'Data root   : {MINTREC_DATA}')
print(f'Output root : {MINTREC_OUTPUTS}')

In [ ]:
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Download MIntRec 2.0 (from HuggingFace)

Pulls the three annotation TSVs + the packed raw videos (`MIntRec2.0_video.tar.gz`, ~9 GB) from `THUIAR/MMLA-Datasets`, then extracts the tar. Public dataset — no token needed (set `HF_TOKEN` only for faster rate limits). Idempotent: HF resumes interrupted downloads, and extraction is skipped once `.mp4` files are present.

In [ ]:
import tarfile
from huggingface_hub import hf_hub_download

HF_REPO = 'THUIAR/MMLA-Datasets'
ANNO_DIR = MINTREC_DATA / 'MIntRec2.0'          # where HF lays the files out
VIDEO_DIR = ANNO_DIR / 'video'                  # where we extract the .mp4 clips

# 1) annotations + video tarball (downloaded into MINTREC_DATA/MIntRec2.0/)
for fn in ['train.tsv', 'dev.tsv', 'test.tsv', 'MIntRec2.0_video.tar.gz']:
    print(f'Fetching {fn} ...')
    hf_hub_download(
        repo_id=HF_REPO, repo_type='dataset',
        filename=f'MIntRec2.0/{fn}', local_dir=str(MINTREC_DATA),
    )

# 2) extract the videos (skip if already done)
already_extracted = VIDEO_DIR.exists() and any(VIDEO_DIR.rglob('*.mp4'))
if already_extracted:
    print('Videos already extracted — skipping.')
else:
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    tar_path = ANNO_DIR / 'MIntRec2.0_video.tar.gz'
    print(f'Extracting {tar_path.name} (~9 GB, be patient) ...')
    with tarfile.open(tar_path, 'r:gz') as tf:
        tf.extractall(VIDEO_DIR)
    print('Extraction done.')

In [ ]:
# Show what landed on disk under the MIntRec2.0 folder.
for p in sorted(ANNO_DIR.iterdir()):
    tag = '/' if p.is_dir() else f'  ({p.stat().st_size/1e6:.1f} MB)'
    print(f'  {p.name}{tag}')
n_mp4 = sum(1 for _ in VIDEO_DIR.rglob('*.mp4')) if VIDEO_DIR.exists() else 0
print(f'\nTotal .mp4 extracted: {n_mp4:,}')

## 3. Load Annotations & Map Raw Videos

The `THUIAR/MMLA-Datasets` TSVs have columns `id, text, label, dimension`, where `id = {dialogue_id}_{utterance_id}` and `label` is the (lower-cased) intent. We keep the official 30-class label list for the inference prompt and map each utterance `id` to its raw `.mp4` (trying both `{id}.mp4` and `dia{d}_utt{u}.mp4` naming).

In [ ]:
# Official 30 in-scope intents — used for the inference prompt + as a canonical reference.
INTENT_LABELS = [
    'Acknowledge', 'Advise', 'Agree', 'Apologise', 'Arrange',
    'Ask for help', 'Asking for opinions', 'Care', 'Comfort', 'Complain',
    'Confirm', 'Criticize', 'Doubt', 'Emphasize', 'Explain',
    'Flaunt', 'Greet', 'Inform', 'Introduce', 'Invite',
    'Joke', 'Leave', 'Oppose', 'Plan', 'Praise',
    'Prevent', 'Refuse', 'Taunt', 'Thank', 'Warn',
]
print(f'{len(INTENT_LABELS)} intent classes')

def load_split(name):
    """Read one MMLA-format MIntRec2.0 split. Columns: id, text, label, dimension.
    id = '{dia}_{utt}'. Adds parsed dia/utt and stripped text/label."""
    df = pd.read_csv(ANNO_DIR / f'{name}.tsv', sep='\t', dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    df['id'] = df['id'].str.strip()
    df['text'] = df['text'].str.strip()
    df['label'] = df['label'].str.strip()
    df['dia'] = df['id'].str.split('_').str[0]
    df['utt'] = df['id'].str.split('_').str[1]
    return df

train_df = load_split('train')
dev_df = load_split('dev')
test_df = load_split('test')
print(f'Columns : {list(train_df.columns)}')
print(f'Train: {len(train_df):,}  |  Dev: {len(dev_df):,}  |  Test: {len(test_df):,}')
print(f'Distinct labels in train ({train_df["label"].nunique()}): {sorted(train_df["label"].unique())}')
train_df.head(4)

In [ ]:
# Build a flexible id -> .mp4 path lookup. THUIAR/MMLA-Datasets names clips
# 'MIntRec2.0_{dia}_{utt}.mp4'; we also try '{id}.mp4' / 'dia{d}_utt{u}.mp4' just in case.
mp4_paths = list(VIDEO_DIR.rglob('*.mp4'))
stem2path = {p.stem: p for p in mp4_paths}
print(f'Indexed {len(mp4_paths):,} .mp4 files')

def find_video(row):
    for key in (f"MIntRec2.0_{row['id']}", row['id'], f"dia{row['dia']}_utt{row['utt']}"):
        if key in stem2path:
            return stem2path[key]
    return None

train_df['video'] = train_df.apply(find_video, axis=1)
have = train_df['video'].notna().mean()
print(f'Train utterances with a matching .mp4: {have*100:.1f}%')
if mp4_paths:
    print(f'Example video filename: {mp4_paths[0].name}')

## 4. Inspect Examples

Peek at a few utterances spanning different intents: print transcript / label, and play the audio track extracted from the raw `.mp4`. The full clip (video + audio) is what we feed the teacher in §6.

In [ ]:
import librosa

# One example per intent, up to 6, that actually has a video on disk.
shown, examples = set(), []
for _, row in train_df.iterrows():
    if row['label'] in shown or row['video'] is None:
        continue
    examples.append(row)
    shown.add(row['label'])
    if len(examples) == 6:
        break

for i, ex in enumerate(examples):
    vpath = ex['video']
    wav, sr = librosa.load(str(vpath), sr=16000, mono=True)  # decodes mp4 audio via ffmpeg
    print(f'\n── Example {i+1} ' + '─' * 30)
    print(f'  id       : {ex["id"]}')
    print(f'  Intent   : {ex["label"]}')
    print(f'  Text     : {ex["text"]}')
    print(f'  Duration : {len(wav)/sr:.2f}s  |  Video: {vpath.name}')
    display(IPyAudio(wav, rate=sr))

## 5. Load Qwen2.5-Omni-3B (4-bit quantised — smoke test)

4-bit NF4 (bitsandbytes) keeps VRAM tiny (~2–3 GB weights), comfortable on an L4 or even a local 6 GB 3060. On a bigger GPU you can drop `quantization_config` and pass `torch_dtype=torch.float16` for the full-precision teacher — hidden-state extraction is identical either way.

In [ ]:
from transformers import (
    Qwen2_5OmniForConditionalGeneration,
    Qwen2_5OmniProcessor,
    BitsAndBytesConfig,
)

MODEL_NAME = 'Qwen/Qwen2.5-Omni-3B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME)
print('Processor loaded.')

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,   # bigger GPU: replace with torch_dtype=torch.float16
    device_map='auto',
    attn_implementation='eager',
)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e9
vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'Model loaded : {MODEL_NAME}  (4-bit NF4)')
print(f'Parameters   : {total_params:.2f}B')
print(f'VRAM used    : {vram_used:.2f} GB')

## 6. Basic Inference Test (video + audio + text → intent)

Feed the raw clip (frames **and** its audio track) plus the transcript, and ask the teacher to pick an intent. `process_mm_info(..., use_audio_in_video=True)` extracts the audio from the same `.mp4` so vision and speech stay aligned.

In [ ]:
from qwen_omni_utils import process_mm_info

sample = examples[0]
vpath = sample['video']

TASK_PROMPT = (
    'You are classifying the speaker\'s intent in a short TV-show clip. '
    'Use the video, the audio, and the transcript. '
    f'Transcript: "{sample["text"]}". '
    'Choose the single best intent from this list and answer with just that label:\n'
    + ', '.join(INTENT_LABELS)
)

conversation = [
    {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful multimodal assistant.'}]},
    {'role': 'user', 'content': [
        {'type': 'video', 'video': str(vpath)},
        {'type': 'text', 'text': TASK_PROMPT},
    ]},
]

text_input = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=True)
inputs = processor(
    text=text_input, audio=audios, images=images, videos=videos,
    return_tensors='pt', padding=True, use_audio_in_video=True,
).to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=32, return_audio=False, use_audio_in_video=True)

prompt_len = inputs['input_ids'].shape[1]
response = processor.batch_decode(generated_ids[:, prompt_len:], skip_special_tokens=True)[0]
print(f'Ground-truth intent : {sample["label"]}')
print(f'Model response      : {response.strip()}')

## 7. Hidden State Extraction (Sanity Check)

These tri-modal hidden states are the distillation target for the tiny student. As in the FSC notebook, the LM backbone lives in `model.thinker` (`generate()` routes through it the same way).

In [ ]:
with torch.no_grad():
    outputs = model.thinker(**inputs, output_hidden_states=True, return_dict=True)

hidden_states = outputs.hidden_states
print(f'Hidden state layers : {len(hidden_states)}  (embedding + {len(hidden_states)-1} transformer blocks)')
print(f'Shape per layer     : {tuple(hidden_states[0].shape)}  [batch, seq_len, hidden_dim]')

mid_idx = len(hidden_states) // 2
mid_repr = hidden_states[mid_idx].float().mean(dim=1)
print(f'\nMid-layer ({mid_idx}) mean-pooled shape : {tuple(mid_repr.shape)}')
vram_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'Peak VRAM (this session) : {vram_peak:.2f} GB')